In [0]:
def ensure_catalog():
    spark.sql("USE CATALOG workspace")
    spark.sql("USE DATABASE ml_layer")

ensure_catalog()

# ── Table 1: Model comparison (all models, both datasets) ────────────────────
model_comparison_data = [
    # Applications
    ("Logistic Regression", "Applications", "Supervised", 0.6748, 0.0224, 0.0538, 0.0175),
    ("Random Forest",       "Applications", "Supervised", 0.7468, 0.0379, 0.0857, 0.0398),
    ("XGBoost",             "Applications", "Supervised", 0.7688, 0.0409, 0.0912, 0.8067),
    ("Autoencoder",         "Applications", "Anomaly",    0.5045, 0.0112, None,   None),
    ("Isolation Forest",    "Applications", "Anomaly",    0.4683, 0.0102, None,   None),
    # Transactions
    ("Logistic Regression", "Transactions", "Supervised", 0.7799, 0.0838, 0.1467, 0.0610),
    ("Random Forest",       "Transactions", "Supervised", 0.8980, 0.6093, 0.6975, 0.0744),
    ("XGBoost",             "Transactions", "Supervised", 0.9114, 0.6325, 0.7052, 0.9380),
    ("Autoencoder",         "Transactions", "Anomaly",    0.6803, 0.0463, None,   None),
    ("Isolation Forest",    "Transactions", "Anomaly",    0.6724, 0.0447, None,   None),
]

import pandas as pd
from pyspark.sql.types import (
    StructType, StructField, StringType,
    DoubleType, IntegerType
)

model_df = pd.DataFrame(model_comparison_data, columns=[
    "model", "dataset", "model_type",
    "roc_auc", "pr_auc", "f1", "best_threshold"
])
spark.createDataFrame(model_df).write.format("delta") \
    .mode("overwrite").option("overwriteSchema", "true") \
    .saveAsTable("workspace.ml_layer.dashboard_model_comparison")

# ── Table 2: Confusion matrix breakdown ──────────────────────────────────────
confusion_data = [
    ("XGBoost", "Applications", 575501, 16825, 5548, 1123),
    ("XGBoost", "Transactions", 2538295, 509, 38184, 46286),
]
confusion_df = pd.DataFrame(confusion_data, columns=[
    "model", "dataset",
    "true_negatives", "false_positives",
    "false_negatives", "true_positives"
])
confusion_df["precision"]     = (
    confusion_df["true_positives"] /
    (confusion_df["true_positives"] + confusion_df["false_positives"])
).round(4)
confusion_df["recall"]        = (
    confusion_df["true_positives"] /
    (confusion_df["true_positives"] + confusion_df["false_negatives"])
).round(4)
confusion_df["fraud_caught"]  = confusion_df["true_positives"]
confusion_df["fraud_missed"]  = confusion_df["false_negatives"]
confusion_df["false_alerts"]  = confusion_df["false_positives"]

spark.createDataFrame(confusion_df).write.format("delta") \
    .mode("overwrite").option("overwriteSchema", "true") \
    .saveAsTable("workspace.ml_layer.dashboard_confusion_matrix")

# ── Table 3: Feature importance (SHAP values from explainability notebook) ───
shap_data = [
    # Applications
    ("address_stability",     "Applications", 0.2899, 1),
    ("log_income",            "Applications", 0.2296, 2),
    ("name_email_similarity", "Applications", 0.2004, 3),
    ("zip_count_4w",          "Applications", 0.1639, 4),
    ("days_since_request",    "Applications", 0.0989, 5),
    ("under_25",              "Applications", 0.0172, 6),
    ("payment_type_index",    "Applications", 0.0001, 7),
    # Transactions
    ("log_amount",                    "Transactions", 0.5760, 1),
    ("merchant_fraud_rate",           "Transactions", 0.1312, 2),
    ("location_city_grouped_index",   "Transactions", 0.1223, 3),
    ("channel_grouped_index",         "Transactions", 0.0608, 4),
    ("device_type_index",             "Transactions", 0.0411, 5),
    ("customer_avg_transaction_amount","Transactions",0.0273, 6),
    ("merchant_category_index",       "Transactions", 0.0197, 7),
    ("transaction_dayofweek",         "Transactions", 0.0094, 8),
    ("transaction_hour",              "Transactions", 0.0072, 9),
    ("amount_balance_ratio",          "Transactions", 0.0049, 10),
]
shap_df = pd.DataFrame(shap_data, columns=[
    "feature", "dataset", "importance", "rank"
])
spark.createDataFrame(shap_df).write.format("delta") \
    .mode("overwrite").option("overwriteSchema", "true") \
    .saveAsTable("workspace.ml_layer.dashboard_feature_importance")

# ── Table 4: Fraud rate summary by dataset ───────────────────────────────────
fraud_summary_data = [
    ("Applications", "Train", 2401003, 26418,  1.10),
    ("Applications", "Test",  598997,  6671,   1.11),
    ("Transactions", "Train", 10493501,339265,  3.23),
    ("Transactions", "Test",  2623274, 84470,   3.22),
]
fraud_summary_df = pd.DataFrame(fraud_summary_data, columns=[
    "dataset", "split", "total_records",
    "fraud_records", "fraud_rate_pct"
])
fraud_summary_df["legit_records"] = (
    fraud_summary_df["total_records"] - fraud_summary_df["fraud_records"]
)
spark.createDataFrame(fraud_summary_df).write.format("delta") \
    .mode("overwrite").option("overwriteSchema", "true") \
    .saveAsTable("workspace.ml_layer.dashboard_fraud_summary")

# ── Table 5: Score distribution buckets for risk tier chart ──────────────────
scored_txn = spark.table("workspace.ml_layer.scored_transactions")
from pyspark.sql.functions import col, count, avg, sum as spark_sum, when

risk_dist = scored_txn.groupBy("risk_tier", "dataset").agg(
    count("*").alias("total"),
    spark_sum("actual_fraud").alias("fraud_count"),
    avg("fraud_score").alias("avg_score")
).toPandas()

spark.createDataFrame(risk_dist).write.format("delta") \
    .mode("overwrite").option("overwriteSchema", "true") \
    .saveAsTable("workspace.ml_layer.dashboard_risk_distribution")

# ── Table 6: Drift detection summary ─────────────────────────────────────────
drift_df = spark.table("workspace.ml_layer.drift_detection_results").toPandas()
drift_df["psi_category"] = pd.cut(
    drift_df["psi"],
    bins   = [0, 0.1, 0.2, float("inf")],
    labels = ["Stable", "Monitor", "Retrain"]
).astype(str)
spark.createDataFrame(drift_df).write.format("delta") \
    .mode("overwrite").option("overwriteSchema", "true") \
    .saveAsTable("workspace.ml_layer.dashboard_drift")

print("✅ All dashboard tables ready")
print("\nTables created:")
spark.sql("SHOW TABLES IN workspace.ml_layer LIKE 'dashboard_*'").show()